In [ ]:
# %% 셀 9: 전체 추론 (Ray 병렬 — 채널마다 worker 재생성)
import os
import glob
import json
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from tqdm.auto import tqdm
import ray

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_PARQUET_DIR = "./data/3_ocr_results"
FRAME_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/4_is_telop_results"
MODEL_SAVE_DIR = "./model/4_is_telop_cnn_models"
CKPT_NAME = "epoch_08_f1_0.9460.pth"

IMG_SIZE = 224
MAX_TEXT_LEN = 64
BATCH_SIZE = 512
THRESHOLD = 0.5
NUM_WORKERS = 8
FRAMES_PER_TASK = 256
OCR_SCORE_THRESHOLD = 0.4

ray.init(num_gpus=1)


# ──────────────────────────────────────────────
# 모델 정의
# ──────────────────────────────────────────────
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, cnn_out=128, gru_hidden=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.cnn = nn.Sequential(
            nn.Conv1d(embed_dim, cnn_out, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(cnn_out, cnn_out, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.gru = nn.GRU(cnn_out, gru_hidden, batch_first=True, bidirectional=True)
        self.out_dim = gru_hidden * 2
    def forward(self, char_ids, text_lens):
        x = self.embed(char_ids).transpose(1, 2)
        x = self.cnn(x).transpose(1, 2)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, text_lens.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        return torch.cat([hidden[0], hidden[1]], dim=-1)


class BboxMLP(nn.Module):
    def __init__(self, in_dim=5, hidden=64, out_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim), nn.ReLU(),
        )
    def forward(self, bbox):
        return self.mlp(bbox)


class TelopMultimodalClassifier(nn.Module):
    def __init__(self, vocab_size, text_out=128, bbox_out=32, fusion_hidden=256):
        super().__init__()
        self.image_enc = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0)
        img_feat_dim = 1280
        self.text_enc = TextEncoder(vocab_size, embed_dim=64, cnn_out=128, gru_hidden=text_out // 2)
        text_feat_dim = self.text_enc.out_dim
        self.bbox_enc = BboxMLP(in_dim=5, hidden=64, out_dim=bbox_out)
        total_dim = img_feat_dim + text_feat_dim + bbox_out
        self.fusion = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(total_dim, fusion_hidden), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(fusion_hidden, 1),
        )
    def forward(self, imgs, char_ids, text_lens, bboxes):
        img_feat = self.image_enc(imgs)
        text_feat = self.text_enc(char_ids, text_lens)
        bbox_feat = self.bbox_enc(bboxes)
        fused = torch.cat([img_feat, text_feat, bbox_feat], dim=-1)
        return self.fusion(fused)


# ──────────────────────────────────────────────
# Ray Worker
# ──────────────────────────────────────────────
@ray.remote(num_gpus=1/NUM_WORKERS, num_cpus=6)
class TelopWorker:
    def __init__(self, ckpt_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        ckpt = torch.load(ckpt_path, map_location=self.device)
        self.vocab_char2idx = ckpt["vocab_char2idx"]
        self.model = TelopMultimodalClassifier(vocab_size=len(self.vocab_char2idx))
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.to(self.device).eval()

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def _crop_rotated_region(self, img, cx, cy, w, h, angle, pad_ratio=0.15):
        h_img, w_img = img.shape[:2]
        w_pad = w * (1 + pad_ratio)
        h_pad = h * (1 + pad_ratio)
        M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w_img, h_img), flags=cv2.INTER_LINEAR)
        x1 = max(0, int(cx - w_pad / 2))
        y1 = max(0, int(cy - h_pad / 2))
        x2 = min(w_img, int(cx + w_pad / 2))
        y2 = min(h_img, int(cy + h_pad / 2))
        crop = rotated[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        return crop

    def _preprocess_crop(self, crop):
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        return self.transform(crop_rgb).numpy()

    def _encode_text(self, text):
        ids = [self.vocab_char2idx.get(ch, 1) for ch in str(text)[:MAX_TEXT_LEN]]
        length = len(ids)
        ids += [0] * (MAX_TEXT_LEN - length)
        return ids, length

    @torch.no_grad()
    def _predict_batch(self, imgs_np, char_ids_np, text_lens_np, bbox_feats_np):
        imgs = torch.from_numpy(imgs_np).to(self.device)
        char_ids = torch.from_numpy(char_ids_np).to(self.device)
        text_lens = torch.from_numpy(text_lens_np).to(self.device)
        bbox_feats = torch.from_numpy(bbox_feats_np).to(self.device)
        with torch.amp.autocast("cuda", enabled=(self.device == "cuda")):
            logits = self.model(imgs, char_ids, text_lens, bbox_feats).squeeze(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        del imgs, char_ids, text_lens, bbox_feats, logits
        torch.cuda.empty_cache()

        return probs

    def process_frames(self, channel, video_name, frame_infos):
        all_imgs = []
        all_char_ids = []
        all_text_lens = []
        all_bbox_feats = []
        crop_map = []

        results = []
        for row_idx, frame_num, ocr_texts, ocr_xywha, ocr_scores in frame_infos:
            n_regions = len(ocr_texts)
            result_idx = len(results)
            results.append({"row_idx": row_idx, "n_regions": n_regions, "probs": {}})

            if not ocr_texts:
                continue

            frame_path = os.path.join(
                FRAME_DIR, channel, video_name, f"frame_{frame_num:08d}.jpg"
            )
            frame = cv2.imread(frame_path)
            if frame is None:
                continue

            for i, (text, bbox, score) in enumerate(zip(ocr_texts, ocr_xywha, ocr_scores)):
                if score < OCR_SCORE_THRESHOLD:   # ← 학습 때와 동일한 필터
                    continue
                cx, cy, w, h, angle = bbox
                crop = self._crop_rotated_region(frame, cx, cy, w, h, angle)
                if crop is None or crop.shape[0] < 4 or crop.shape[1] < 4:
                    continue

                all_imgs.append(self._preprocess_crop(crop))
                cid, tlen = self._encode_text(text)
                all_char_ids.append(cid)
                all_text_lens.append(tlen)
                all_bbox_feats.append(np.array([
                    cx / 1920.0, cy / 1080.0, w / 1920.0, h / 1080.0, angle / 180.0,
                ], dtype=np.float32))
                crop_map.append((result_idx, i))

        if all_imgs:
            imgs_np = np.stack(all_imgs)
            char_ids_np = np.array(all_char_ids, dtype=np.int64)
            text_lens_np = np.array(all_text_lens, dtype=np.int64)
            bbox_feats_np = np.stack(all_bbox_feats)

            all_probs = []
            for start in range(0, len(all_imgs), BATCH_SIZE):
                end = min(start + BATCH_SIZE, len(all_imgs))
                probs = self._predict_batch(
                    imgs_np[start:end], char_ids_np[start:end],
                    text_lens_np[start:end], bbox_feats_np[start:end],
                )
                all_probs.append(probs)
            all_probs = np.concatenate(all_probs)

            for j, (result_idx, region_idx) in enumerate(crop_map):
                results[result_idx]["probs"][region_idx] = float(all_probs[j])

        return results


# ──────────────────────────────────────────────
# 전체 추론 — 채널마다 worker 생성/삭제
# ──────────────────────────────────────────────
ckpt_path = os.path.join(MODEL_SAVE_DIR, CKPT_NAME)

total_parquets = 0
total_frames = 0
total_regions = 0

for channel_entry in sorted(os.scandir(OCR_PARQUET_DIR), key=lambda e: e.name):
    if not channel_entry.is_dir():
        continue
    channel = channel_entry.name
    out_channel_dir = os.path.join(OUTPUT_DIR, channel)
    os.makedirs(out_channel_dir, exist_ok=True)

    pq_paths = sorted([
        os.path.join(channel_entry.path, f)
        for f in os.listdir(channel_entry.path)
        if f.endswith(".parquet")
    ])


    pq_meta = {}
    task_to_pq = {}
    all_tasks = []
    task_counter = 0

    for pq_path in pq_paths:
        video_name = os.path.basename(pq_path)[:-8]
        out_path = os.path.join(out_channel_dir, os.path.basename(pq_path))

        if os.path.exists(out_path):
            continue

        df = pd.read_parquet(pq_path)
        pq_key = pq_path

        frame_infos = []
        for df_idx, row in df.iterrows():
            frame_num = int(row["frame_num"])
            ocr_texts = json.loads(row["ocr_texts"]) if isinstance(row["ocr_texts"], str) else row["ocr_texts"]
            ocr_xywha = json.loads(row["ocr_xywha"]) if isinstance(row["ocr_xywha"], str) else row["ocr_xywha"]
            ocr_scores = json.loads(row["ocr_scores"]) if isinstance(row["ocr_scores"], str) else row["ocr_scores"]
            frame_infos.append((df_idx, frame_num, ocr_texts, ocr_xywha, ocr_scores))

        n_tasks = 0
        for i in range(0, len(frame_infos), FRAMES_PER_TASK):
            chunk = frame_infos[i:i + FRAMES_PER_TASK]
            n_tasks += 1

        pq_meta[pq_key] = {
            "df": df,
            "out_path": out_path,
            "pending": n_tasks,
            "results": [],
            "frame_infos": frame_infos,
        }

    # 이 채널에 처리할 게 없으면 skip
    if not pq_meta:
        continue

    # 채널 시작: worker 생성
    workers = [TelopWorker.remote(ckpt_path) for _ in range(NUM_WORKERS)]

    # 태스크 제출
    for pq_key, meta in pq_meta.items():
        video_name = os.path.basename(pq_key)[:-8]
        frame_infos = meta["frame_infos"]

        for i in range(0, len(frame_infos), FRAMES_PER_TASK):
            chunk = frame_infos[i:i + FRAMES_PER_TASK]
            worker = workers[task_counter % NUM_WORKERS]
            task = worker.process_frames.remote(channel, video_name, chunk)
            task_to_pq[task] = pq_key
            all_tasks.append(task)
            task_counter += 1

        del meta["frame_infos"]  # 제출 후 불필요

    pbar = tqdm(total=len(all_tasks), desc=f"📂 {channel}", leave=True)
    remaining = list(all_tasks)

    while remaining:
        ready, remaining = ray.wait(remaining, num_returns=min(8, len(remaining)), timeout=1.0)
        if not ready:
            continue

        for task in ready:
            pq_key = task_to_pq[task]
            batch_results = ray.get(task)
            meta = pq_meta[pq_key]
            meta["results"].extend(batch_results)
            meta["pending"] -= 1
            pbar.update(1)
            del task_to_pq[task]

            if meta["pending"] == 0:
                df = meta["df"]
                all_results = meta["results"]
                all_results.sort(key=lambda x: x["row_idx"])

                is_telop_col = []
                telop_p_col = []

                for r in all_results:
                    n_regions = r["n_regions"]
                    probs_dict = r["probs"]

                    if n_regions == 0:
                        is_telop_col.append(json.dumps([]))
                        telop_p_col.append(json.dumps([]))
                    else:
                        labels = [None] * n_regions
                        probs_list = [None] * n_regions
                        for region_idx in range(n_regions):
                            if region_idx in probs_dict:
                                p = probs_dict[region_idx]
                                probs_list[region_idx] = round(p, 4)
                                labels[region_idx] = bool(p >= THRESHOLD)
                        is_telop_col.append(json.dumps(labels))
                        telop_p_col.append(json.dumps(probs_list))

                out_df = pd.DataFrame({
                    "frame_num": df["frame_num"],
                    "is_telop": is_telop_col,
                    "telop_p": telop_p_col,
                })
                out_df.to_parquet(meta["out_path"], index=False)

                total_frames += len(df)
                total_regions += sum(r["n_regions"] for r in all_results)
                total_parquets += 1

                del pq_meta[pq_key]

    pbar.close()

    # 채널 끝: 정리
    pq_meta.clear()
    all_tasks.clear()
    task_to_pq.clear()
    for w in workers:
        ray.kill(w)
    del workers

ray.shutdown()

print(f"\n✅ 추론 완료")
print(f"   parquet: {total_parquets:,}개")
print(f"   프레임: {total_frames:,}개")
print(f"   regions: {total_regions:,}개")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-09 13:38:32,458	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-09 13:38:34,698	INFO worker.py:2013 -- Started a local Ray instance.
/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
📂 DinoCore TUBA:  54%|█████▎    | 141/263 [01:39<00:57,  2.13it/s]

In [6]:
# %% 셀 10: 추론 결과 검증 (32코어 병렬)
import os
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

OCR_PARQUET_DIR = "./data/3_ocr_results"
OUTPUT_DIR = "./data/4_is_telop_results"
NUM_VERIFY_WORKERS = 32


def verify_one(src_path, out_path, channel, video_name):
    result = {
        "channel": channel,
        "video_name": video_name,
        "frames": 0,
        "regions": 0,
        "telop": 0,
        "scene": 0,
        "none": 0,
        "row_mismatch": None,
        "length_mismatches": [],
        "none_examples": [],
    }

    if not os.path.exists(out_path):
        result["missing"] = True
        return result
    result["missing"] = False

    df_src = pd.read_parquet(src_path)
    df_out = pd.read_parquet(out_path)

    if len(df_src) != len(df_out):
        result["row_mismatch"] = (len(df_src), len(df_out))
        return result

    if not (df_src["frame_num"].values == df_out["frame_num"].values).all():
        result["row_mismatch"] = ("frame_num 순서 불일치", "")
        return result

    result["frames"] = len(df_out)

    for i in range(len(df_out)):
        src_texts = json.loads(df_src.iloc[i]["ocr_texts"]) if isinstance(df_src.iloc[i]["ocr_texts"], str) else df_src.iloc[i]["ocr_texts"]
        is_telop = json.loads(df_out.iloc[i]["is_telop"])
        telop_p = json.loads(df_out.iloc[i]["telop_p"])
        n_src = len(src_texts)

        if len(is_telop) != n_src or len(telop_p) != n_src:
            result["length_mismatches"].append((
                df_out.iloc[i]["frame_num"], n_src, len(is_telop), len(telop_p),
            ))
            continue

        for j in range(n_src):
            result["regions"] += 1
            if is_telop[j] is None or telop_p[j] is None:
                result["none"] += 1
                if len(result["none_examples"]) < 3:
                    result["none_examples"].append((
                        df_out.iloc[i]["frame_num"], j, is_telop[j], telop_p[j],
                    ))
            elif is_telop[j]:
                result["telop"] += 1
            else:
                result["scene"] += 1

    return result


# ── 검증 대상 수집 (os.listdir 사용) ──
tasks = []
for channel_entry in sorted(os.scandir(OCR_PARQUET_DIR), key=lambda e: e.name):
    if not channel_entry.is_dir():
        continue
    channel = channel_entry.name
    for fname in sorted(os.listdir(channel_entry.path)):
        if not fname.endswith(".parquet"):
            continue
        video_name = fname[:-8]
        src_path = os.path.join(channel_entry.path, fname)
        out_path = os.path.join(OUTPUT_DIR, channel, fname)
        tasks.append((src_path, out_path, channel, video_name))

print(f"📁 검증 대상: {len(tasks):,}개 parquet")

# ── 병렬 실행 ──
total_pq = 0
total_frames = 0
total_regions = 0
total_none = 0
total_telop = 0
total_scene = 0
missing_pq = []
row_mismatch = []
length_mismatch = []
none_examples = []

with ProcessPoolExecutor(max_workers=NUM_VERIFY_WORKERS) as executor:
    futures = {executor.submit(verify_one, *t): t for t in tasks}

    for fut in tqdm(as_completed(futures), total=len(futures), desc="🔍 검증"):
        r = fut.result()
        key = f"{r['channel']}/{r['video_name']}"

        if r.get("missing"):
            missing_pq.append(key)
            continue

        if r["row_mismatch"] is not None:
            row_mismatch.append((key, *r["row_mismatch"]) if isinstance(r["row_mismatch"], tuple) else (key, r["row_mismatch"]))
            continue

        total_pq += 1
        total_frames += r["frames"]
        total_regions += r["regions"]
        total_telop += r["telop"]
        total_scene += r["scene"]
        total_none += r["none"]

        for lm in r["length_mismatches"]:
            length_mismatch.append((key, *lm))

        for ne in r["none_examples"]:
            if len(none_examples) < 20:
                none_examples.append((key, *ne))

# ──────────────────────────────────────────────
# 결과 출력
# ──────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("📊 추론 결과 검증")
print("=" * 60)

print(f"\n📁 parquet 파일")
print(f"  처리 완료: {total_pq:,}개")
print(f"  누락: {len(missing_pq):,}개")
if missing_pq:
    for p in missing_pq[:10]:
        print(f"    ❌ {p}")
    if len(missing_pq) > 10:
        print(f"    ... +{len(missing_pq)-10}개")

print(f"\n📏 행/순서 불일치")
print(f"  불일치: {len(row_mismatch):,}개")
if row_mismatch:
    for rm in row_mismatch[:10]:
        print(f"    ❌ {rm}")

print(f"\n📐 리스트 길이 불일치")
print(f"  불일치: {len(length_mismatch):,}개")
if length_mismatch:
    for lm in length_mismatch[:10]:
        print(f"    ❌ {lm}")

print(f"\n📈 Region 통계")
print(f"  총 region: {total_regions:,}")
print(f"  telop: {total_telop:,} ({total_telop/max(total_regions,1):.1%})")
print(f"  scene_text: {total_scene:,} ({total_scene/max(total_regions,1):.1%})")
print(f"  None (예측 실패): {total_none:,} ({total_none/max(total_regions,1):.1%})")

if none_examples:
    print(f"\n  None 샘플 (최대 20개):")
    for ex in none_examples:
        print(f"    {ex}")

print(f"\n{'=' * 60}")
issues = len(missing_pq) + len(row_mismatch) + len(length_mismatch)
if issues == 0 and total_none == 0:
    print("✅ 전체 검증 통과 — 문제 없음")
elif issues == 0 and total_none > 0:
    print(f"⚠️ 구조 정상, 예측 실패 {total_none:,}건 ({total_none/total_regions:.2%})")
else:
    print(f"❌ 문제 발견: 누락 {len(missing_pq)}, 행불일치 {len(row_mismatch)}, 길이불일치 {len(length_mismatch)}, None {total_none:,}")

📁 검증 대상: 66,400개 parquet


🔍 검증: 100%|██████████| 66400/66400 [01:48<00:00, 614.53it/s] 



📊 추론 결과 검증

📁 parquet 파일
  처리 완료: 66,400개
  누락: 0개

📏 행/순서 불일치
  불일치: 0개

📐 리스트 길이 불일치
  불일치: 0개

📈 Region 통계
  총 region: 133,596,393
  telop: 92,875,783 (69.5%)
  scene_text: 40,720,610 (30.5%)
  None (예측 실패): 0 (0.0%)

✅ 전체 검증 통과 — 문제 없음
